# 00 · Setup and data

This is the first of four notebooks that walk through **GPU_HYPE**, the GPU-assisted, multi-objective calibration and probabilistic forecasting tool for the [HYPE](https://hypeweb.smhi.se/) hydrological model published alongside the paper.

| Notebook | Topic |
|---|---|
| **00 · Setup and data** *(this one)* | environment, OpenCL check, the HYPE folder and the data formats |
| [01 · Calibration](01_calibration.ipynb) | the calibration tool: build `HYPE` + `Error` + `GPU_PSO` and run a short `fit` |
| [02 · Training and predicting](02_training_and_predicting.ipynb) | load the pre-trained model and forecast a probabilistic ensemble |
| [03 · Results](03_results.ipynb) | hydrographs with uncertainty bands, reliability, and skill metrics |

All four run with the working directory at the **repository root** and use the bundled **Türkheim** catchment under [`examples/`](../examples).

> **Requirements.** A runnable end-to-end pipeline needs **Windows** (the HYPE model is a `.exe`) and a working **OpenCL** platform (a GPU, or a CPU OpenCL runtime). See the [README](../README.md#system-requirements).

In [ ]:
# --- Bootstrap: make `import gpu` and the examples/ paths resolve from the repo root ---
import os, sys
from pathlib import Path

def find_repo_root(start=None):
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "gpu.py").exists():
            return p
    raise RuntimeError("Could not find the repo root (no gpu.py found above %s)" % start)

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

## 1 · Check the OpenCL platform

GPU_HYPE evaluates its error metric (and the optional ANN model) with **OpenCL** kernels. Importing the optimiser builds an OpenCL context, so at least one OpenCL platform must be available. The list below should be non-empty.

In [ ]:
import pyopencl as cl

platforms = cl.get_platforms()
assert platforms, "No OpenCL platform found - install a GPU/CPU OpenCL runtime."
for p in platforms:
    print("Platform:", p.name)
    for d in p.get_devices():
        print("   device:", d.name, "|", cl.device_type.to_string(d.type))

## 2 · The HYPE folder

A **HYPE folder** is a standard HYPE setup directory. The wrapper copies it several times into a temporary directory and runs those copies in parallel, one per population sub-batch. The bundled example is the **Türkheim** sub-catchment (outlet subbasin `50675`).

Key files:

| File | Role |
|---|---|
| `info.txt` | simulation dates (`bdate`/`cdate`/`edate`) and `basinoutput` settings |
| `par.txt` | model parameters (the calibration targets live here) |
| `GeoData.txt`, `GeoClass.txt` | subbasin topology and land-use classes |
| `Pobs.txt`, `Tobs.txt`, `TMINobs.txt`, `TMAXobs.txt` | meteorological forcing |
| `HYPEwithoutPopup4All.exe` | the HYPE executable the wrapper calls |

In [ ]:
HYPE_FOLDER = Path("examples/set7_germany_tuerkheim")
for f in sorted(HYPE_FOLDER.iterdir()):
    print(f"{f.stat().st_size:>10,d}  {f.name}")

In [ ]:
# Simulation window and output subbasin are declared in info.txt
for line in (HYPE_FOLDER / "info.txt").read_text().splitlines():
    if line.startswith(("bdate", "cdate", "edate", "resultdir")) or "subbasins" in line:
        print(line)

## 3 · Observed discharge

The observations used to score the calibration are the daily discharge at the outlet (subbasin `50675`). They are stored tab-separated with a `Date` column.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

obs = pd.read_csv("examples/Qobs.txt", sep="\t", header=0,
                  index_col="Date", parse_dates=True)
discharge = obs["50675"]
print("records:", len(discharge), "| from", discharge.index.min().date(), "to", discharge.index.max().date())
discharge.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
discharge.loc["1980":"1999"].plot(ax=ax, lw=0.7)
ax.set_ylabel("Discharge [m\u00b3/s]")
ax.set_xlabel("")
ax.set_title("Observed discharge \u2014 T\u00fcrkheim (subbasin 50675), calibration period")
plt.tight_layout()

## 4 · Meteorological forcing

HYPE reads its forcing (precipitation, temperature) directly from the HYPE folder; you do not pass it to the optimiser. We load it here only to understand the inputs and to overlay it on the hydrograph later (notebook 03).

In [ ]:
prec = pd.read_csv(HYPE_FOLDER / "Pobs.txt", sep="\t", header=0,
                   index_col="DATE", parse_dates=True)
temp = pd.read_csv(HYPE_FOLDER / "Tobs.txt", sep="\t", header=0,
                   index_col="DATE", parse_dates=True)
print("Precipitation columns (subbasins):", list(prec.columns))
print("Temperature columns (subbasins):", list(temp.columns))
prec.head()

In [ ]:
fig, (axp, axt) = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
prec.loc["1985"].mean(axis=1).plot(ax=axp, color="tab:blue")
axp.set_ylabel("Precip [mm]"); axp.invert_yaxis()
temp.loc["1985"].mean(axis=1).plot(ax=axt, color="tab:red")
axt.set_ylabel("Temp [\u00b0C]"); axt.set_xlabel("")
axp.set_title("Catchment-average forcing (1985)")
plt.tight_layout()

## Next

Everything is in place. Continue with [**01 · Calibration**](01_calibration.ipynb) to set up the optimiser and run a short calibration on this catchment.